In [12]:
#Nicole Jae Olandria

In [1]:
#imports

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [2]:
#Reading in earthquake data

data = pd.read_csv("https://raw.githubusercontent.com/nicole-jae/Earthquake-Prediction-Model/refs/heads/main/quake%20data.csv")
data.head(5)

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2020-01-01T07:04:14.310Z,36.411500,-98.144667,7.80,3.08,ml,59.0,64.0,0.00000,0.12,...,2020-03-21T17:13:07.040Z,"4 km NE of Meno, Oklahoma",earthquake,NaN,0.40,0.190,19.0,reviewed,ok,ok
1,2020-01-01T10:38:35.620Z,33.811333,-117.689667,9.79,3.16,ml,153.0,21.0,0.05963,0.22,...,2024-10-14T17:48:51.290Z,"11km ENE of North Tustin, CA",earthquake,0.12,0.28,0.167,240.0,reviewed,ci,ci
2,2020-01-01T13:57:05.760Z,35.983333,-117.315500,2.67,2.71,ml,48.0,53.0,0.09582,0.18,...,2024-10-14T22:53:28.180Z,"25km NNE of Trona, CA",earthquake,0.18,0.55,0.103,27.0,reviewed,ci,ci
3,2020-01-01T18:08:04.678Z,42.206100,-98.723300,9.81,2.80,mb_lg,NaN,54.0,0.80700,1.09,...,2020-03-21T17:13:22.040Z,"2 km E of Chambers, Nebraska",earthquake,1.90,5.20,0.088,34.0,reviewed,us,us
4,2020-01-01T18:41:29.870Z,40.396667,-124.930500,4.12,2.51,md,16.0,296.0,0.45940,0.12,...,2020-03-21T17:13:03.040Z,"55km W of Petrolia, CA",earthquake,1.58,1.14,0.225,18.0,reviewed,nc,nc


In [3]:
#Cleaning up data to show date and location only

columns = ['time', 'latitude', 'longitude', 'place']
data_cleaned = data[columns]
data_cleaned.head(5)

,time,latitude,longitude,place
0,2020-01-01T07:04:14.310Z,36.411500,-98.144667,"4 km NE of Meno, Oklahoma"
1,2020-01-01T10:38:35.620Z,33.811333,-117.689667,"11km ENE of North Tustin, CA"
2,2020-01-01T13:57:05.760Z,35.983333,-117.315500,"25km NNE of Trona, CA"
3,2020-01-01T18:08:04.678Z,42.206100,-98.723300,"2 km E of Chambers, Nebraska"
4,2020-01-01T18:41:29.870Z,40.396667,-124.930500,"55km W of Petrolia, CA"


In [4]:
#Making the time data easier to read

data_cleaned['time'] = pd.to_datetime(data_cleaned['time'])
data_cleaned['year'] = data_cleaned['time'].dt.year
data_cleaned['month'] = data_cleaned['time'].dt.month
data_cleaned['day'] = data_cleaned['time'].dt.day
data_cleaned['day_of_week'] = data_cleaned['time'].dt.dayofweek #Monday=0, Sunday=6
data_cleaned['hour'] = data_cleaned['time'].dt.hour
data_cleaned.head(5)

C:\Users\nicol\AppData\Local\Temp\ipykernel_18192\689198931.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['time'] = pd.to_datetime(data_cleaned['time'])
C:\Users\nicol\AppData\Local\Temp\ipykernel_18192\689198931.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['year'] = data_cleaned['time'].dt.year
C:\Users\nicol\AppData\Local\Temp\ipykernel_18192\689198931.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

,time,latitude,longitude,place,year,month,day,day_of_week,hour
0,2020-01-01 07:04:14.310000+00:00,36.411500,-98.144667,"4 km NE of Meno, Oklahoma",2020,1,1,2,7
1,2020-01-01 10:38:35.620000+00:00,33.811333,-117.689667,"11km ENE of North Tustin, CA",2020,1,1,2,10
2,2020-01-01 13:57:05.760000+00:00,35.983333,-117.315500,"25km NNE of Trona, CA",2020,1,1,2,13
3,2020-01-01 18:08:04.678000+00:00,42.206100,-98.723300,"2 km E of Chambers, Nebraska",2020,1,1,2,18
4,2020-01-01 18:41:29.870000+00:00,40.396667,-124.930500,"55km W of Petrolia, CA",2020,1,1,2,18


In [5]:
#Simplifying the place column

def location_simple(place_str):
  if 'of' in place_str:
    return place_str.split('of',1)[1].strip()
  else:
    return place_str.strip()

data_cleaned['place'] = data_cleaned['place'].apply(location_simple)
data_cleaned.head(5)

C:\Users\nicol\AppData\Local\Temp\ipykernel_18192\2801251691.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['place'] = data_cleaned['place'].apply(location_simple)


,time,latitude,longitude,place,year,month,day,day_of_week,hour
0,2020-01-01 07:04:14.310000+00:00,36.411500,-98.144667,"Meno, Oklahoma",2020,1,1,2,7
1,2020-01-01 10:38:35.620000+00:00,33.811333,-117.689667,"North Tustin, CA",2020,1,1,2,10
2,2020-01-01 13:57:05.760000+00:00,35.983333,-117.315500,"Trona, CA",2020,1,1,2,13
3,2020-01-01 18:08:04.678000+00:00,42.206100,-98.723300,"Chambers, Nebraska",2020,1,1,2,18
4,2020-01-01 18:41:29.870000+00:00,40.396667,-124.930500,"Petrolia, CA",2020,1,1,2,18


In [6]:
#Counting earthquake instances at the location

data_cleaned['date'] = data_cleaned['time'].dt.date
daily_earthquakes = data_cleaned.groupby(['date', 'place']).size().reset_index(name='earthquake_count')
daily_earthquakes['earthquake occured'] = (daily_earthquakes['earthquake_count'] > 0).astype(int)
daily_earthquakes.head(5)

C:\Users\nicol\AppData\Local\Temp\ipykernel_18192\1894194682.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_cleaned['date'] = data_cleaned['time'].dt.date


,date,place,earthquake_count,earthquake occured
0,2020-01-01,"Chambers, Nebraska",1,1
1,2020-01-01,"Meno, Oklahoma",1,1
2,2020-01-01,"North Tustin, CA",1,1
3,2020-01-01,"Petrolia, CA",1,1
4,2020-01-01,"Trona, CA",1,1


In [7]:
#Need instances where no earthquake occurred for the model

all_dates = data_cleaned['date'].unique()
all_locations = data_cleaned['place'].unique()
all_combinations = pd.MultiIndex.from_product([all_dates, all_locations], names=['date', 'place']).to_frame(index=False)

full_daily_earthquakes = pd.merge(all_combinations, daily_earthquakes[['date', 'place', 'earthquake_count']], on=['date', 'place'], how='left')
full_daily_earthquakes['earthquake_count'] = full_daily_earthquakes['earthquake_count'].fillna(0).astype(int)
full_daily_earthquakes['earthquake_occurred'] = (full_daily_earthquakes['earthquake_count'] > 0).astype(int)
daily_earthquakes = full_daily_earthquakes.copy()

daily_earthquakes.head(10)

,date,place,earthquake_count,earthquake_occurred
0,2020-01-01,"Meno, Oklahoma",1,1
1,2020-01-01,"North Tustin, CA",1,1
2,2020-01-01,"Trona, CA",1,1
3,2020-01-01,"Chambers, Nebraska",1,1
4,2020-01-01,"Petrolia, CA",1,1
5,2020-01-01,"Morgan Hill, CA",0,0
6,2020-01-01,"Mammoth Lakes, CA",0,0
7,2020-01-01,"Port Hueneme, CA",0,0
8,2020-01-01,"Whites City, New Mexico",0,0
9,2020-01-01,"Pinnacles, CA",0,0


In [8]:
#Building the earthquake prediction training model

daily_earthquakes['month'] = pd.to_datetime(daily_earthquakes['date']).dt.month
daily_earthquakes['day'] = pd.to_datetime(daily_earthquakes['date']).dt.day
daily_earthquakes['day_of_week'] = pd.to_datetime(daily_earthquakes['date']).dt.dayofweek

le = LabelEncoder()
daily_earthquakes['location_encoded'] = le.fit_transform(daily_earthquakes['place'])

X = daily_earthquakes[['month', 'day', 'day_of_week', 'location_encoded']]
y = daily_earthquakes['earthquake_occurred']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Testing target shape: {y_test.shape}")

Training features shape: (2588751, 4)
Training target shape: (2588751,)
Testing features shape: (1109466, 4)
Testing target shape: (1109466,)


In [9]:
#Training the model

model = LogisticRegression(random_state=42, solver='liblinear')
model.fit(X_train, y_train)

y_prediction = model.predict(X_test)
y_pred_probability = model.predict_proba(X_test)[:, 1]

print(f"Model Accuracy: {accuracy_score(y_test, y_prediction)}")
print("\nClassification Report:\n", classification_report(y_test, y_prediction))

results = X_test.copy()
results['actual_earthquake'] = y_test
results['predicted_earthquake'] = y_prediction
results['prediction_probability'] = y_pred_probability

results.head(10)

Model Accuracy: 0.9965794355122194

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00   1105671
           1       0.00      0.00      0.00      3795

    accuracy                           1.00   1109466
   macro avg       0.50      0.50      0.50   1109466
weighted avg       0.99      1.00      0.99   1109466



C:\Users\nicol\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicol\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\nicol\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,month,day,day_of_week,location_encoded,actual_earthquake,predicted_earthquake,prediction_probability
518180,11,2,0,1610,0,0,0.005840
3218574,3,24,0,74,0,0,0.001843
2235134,8,19,5,533,0,0,0.002467
722700,3,3,2,1439,0,0,0.005229
2003361,4,2,6,26,0,0,0.001727
13783,1,9,3,1396,0,0,0.005035
2149161,6,28,2,426,0,0,0.002328
2165109,7,8,5,111,0,0,0.001820
714455,2,26,4,442,0,0,0.002367
1626171,8,21,6,809,0,0,0.003012


In [11]:
#Saving the models for easy loading

joblib.dump(model, 'logistic_regression_model.pk1')
print("Logistic Regression Model saved as 'logistic_regression_model'")

joblib.dump(le, 'label_encoder.pk1')
print ("Label Encoder saved as 'label_encoder.pk1'")

data_cleaned.to_csv('cleaned_quake_data.csv', index=False)
print("Cleaned data saved as 'cleaned_quake_data.csv'")
            

Logistic Regression Model saved as 'logistic_regression_model'
Label Encoder saved as 'label_encoder.pk1'
Cleaned data saved as 'cleaned_quake_data.csv'
